# Isolated Level 0 — single-seed diagnostics

GPT-2 BPE AdamW/Muon baseline on pinned FineWeb-Edu.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv('NANOGPT_LEVEL0_RESULTS_ROOT', '/tmp/nanogpt-level0-gpt2/results'))
OPTIMIZER = os.getenv('NANOGPT_LEVEL0_NOTEBOOK_OPTIMIZER', 'adamw')
SEED = int(os.getenv('NANOGPT_LEVEL0_NOTEBOOK_SEED', '1337'))
RUN = ROOT / f'{OPTIMIZER}_seed_{SEED}'

required = ['manifest.json', 'metrics.csv', 'run_complete.json', 'final_metrics.json', 'selected_checkpoint_metrics.json']
missing = [name for name in required if not (RUN / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing run artifacts in {RUN}: {missing}')

manifest = json.loads((RUN / 'manifest.json').read_text())
completion = json.loads((RUN / 'run_complete.json').read_text())
final_metrics = json.loads((RUN / 'final_metrics.json').read_text())
selected_metrics = json.loads((RUN / 'selected_checkpoint_metrics.json').read_text())
metrics = pd.read_csv(RUN / 'metrics.csv')

pd.DataFrame([{
    'run': str(RUN),
    'parameters': manifest['parameter_count'],
    'optimizer_steps': completion['optimizer_steps'],
    'tokens_seen': completion['tokens_seen'],
    'best_validation_step': completion['best_validation_step'],
    'best_validation_loss': completion['best_validation_loss'],
    'selected_test_loss': completion['selected_test_loss'],
    'final_test_loss': completion['final_test_loss'],
    'weightwatcher_failures': completion['weightwatcher_failures'],
}])

In [ ]:
for metric, ylabel in [
    ('loss', 'cross-entropy loss'),
    ('perplexity', 'perplexity'),
    ('accuracy', 'exact next-token accuracy (%)'),
]:
    plt.figure(figsize=(10, 5))
    for split in ('train', 'val'):
        values = metrics[f'{split}_{metric}']
        if metric == 'accuracy':
            values = 100 * values
        plt.plot(metrics['tokens_seen'], values, label=split)
    plt.xlabel('training tokens')
    plt.ylabel(ylabel)
    plt.title(f'{OPTIMIZER} seed {SEED}: {metric}')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(metrics['tokens_seen'], metrics['val_generalization_gap'])
plt.axhline(0.0, linewidth=1, linestyle='--')
plt.xlabel('training tokens')
plt.ylabel('validation loss − training loss')
plt.title(f'{OPTIMIZER} seed {SEED}: generalization gap')
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(metrics['step'], metrics['learning_rate'], label='base learning rate')
plt.plot(metrics['step'], metrics['min_group_learning_rate'], linestyle='--', label='minimum parameter-group LR')
plt.plot(metrics['step'], metrics['max_group_learning_rate'], linestyle=':', label='maximum parameter-group LR')
plt.xlabel('optimizer step')
plt.ylabel('learning rate')
plt.title(f'{OPTIMIZER} seed {SEED}: learning-rate schedule')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

In [ ]:
checkpoint_comparison = pd.DataFrame([
    {
        'checkpoint': 'validation-selected',
        'step': selected_metrics['selected_step'],
        'validation_loss': selected_metrics['validation_loss'],
        'test_loss': selected_metrics['test_loss'],
        'test_perplexity': selected_metrics['test_perplexity'],
        'test_accuracy_percent': 100 * selected_metrics['test_accuracy'],
    },
    {
        'checkpoint': 'final',
        'step': final_metrics['step'],
        'validation_loss': final_metrics['validation_loss'],
        'test_loss': final_metrics['test_loss'],
        'test_perplexity': final_metrics['test_perplexity'],
        'test_accuracy_percent': 100 * final_metrics['test_accuracy'],
    },
])
checkpoint_comparison

In [ ]:
ww_files = sorted(RUN.glob('weightwatcher_step_*.csv'))
if not ww_files:
    print('No successful WeightWatcher measurements found.')
else:
    ww = pd.concat([pd.read_csv(path) for path in ww_files], ignore_index=True)
    ww['alpha'] = pd.to_numeric(ww.get('alpha'), errors='coerce')
    ww['step'] = pd.to_numeric(ww['step'], errors='coerce')
    layer_column = next((name for name in ('source_layer', 'longname', 'name', 'layer_id') if name in ww), None)
    if layer_column is None:
        raise KeyError('WeightWatcher output has no layer identifier column')
    valid = ww[np.isfinite(ww['alpha'])].copy()
    plt.figure(figsize=(12, 7))
    for layer, frame in valid.groupby(layer_column):
        frame = frame.sort_values('step')
        plt.plot(frame['step'], frame['alpha'], marker='o', linewidth=1, markersize=3, label=str(layer))
    plt.axhline(2.0, linestyle='--', linewidth=1, label='alpha = 2')
    plt.xlabel('optimizer step')
    plt.ylabel('WeightWatcher alpha')
    plt.title(f'{OPTIMIZER} seed {SEED}: transformer-matrix alpha trajectories')
    plt.grid(alpha=0.25)
    plt.legend(fontsize=7, ncol=2, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    display_columns = [column for column in ('step', layer_column, 'alpha', 'D', 'xmin', 'num_evals') if column in valid]
    display(valid[display_columns].sort_values(['step', layer_column]).tail(30))